# Install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install openai groq python-dotenv

# Import libraries

In [ ]:
from google.colab import files
from dotenv import load_dotenv
import os
import pandas as pd
import time
import copy
# Select appropriate library based on the model provider's API
# from openai import OpenAI       # OpenRouter
from groq import Groq           # Groq

# Read file questions 2 turns

In [ ]:
# Adjust file path to match the target environment
init_df = pd.read_csv('drive/MyDrive/ViJudge/data/questions/questions_2_turns.csv')
init_df.head(2)

# Preprocessing

In [ ]:
# Remove unnecessary attributes
df = init_df.drop(columns=['Vietnamese_reference'])

In [ ]:
# Split into separate DataFrames for turn 1 and turn 2 questions
turn1_df = df[df['turn'] == 1]
turn2_df = df[df['turn'] == 2]

In [ ]:
turn1_df = turn1_df.drop(columns=['turn'])
turn2_df = turn2_df.drop(columns=['turn'])

In [ ]:
turn1_df = turn1_df.rename(columns={'Vietnamese_content': 'content_turn1'})
turn2_df = turn2_df.rename(columns={'Vietnamese_content': 'content_turn2'})

In [ ]:
# Merge 2 df on 'question_id' and 'category' to create a combined df with both turns
merged_df = pd.merge(turn1_df, turn2_df, on=['question_id', 'category'])
merged_df.tail(2)

In [ ]:
# Convert to list of questions
questions = merged_df.to_dict('records')

In [ ]:
# Check
questions[0]

# Upload file .env to Colab

In [ ]:
uploaded_file = files.upload()

# Get API Key

In [ ]:
load_dotenv()

# Set API key corresponding to the selected library
# openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

# if not openrouter_api_key:
    # raise ValueError("OPENROUTER_API_KEY not found")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found")

# Connect with OpenRouter/Groq

In [ ]:
# openrouter_client = OpenAI(
    # api_key = openrouter_api_key,
    # base_url = "https://openrouter.ai/api/v1",
# )

groq_client = Groq(
    api_key = groq_api_key,
)

# Collect LLM responses

In [ ]:
groq_models = ['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b']
openrouter_models = ['deepseek/deepseek-chat-v3-0324:free']

In [ ]:
def gen_model_answer(model, client, list_questions):
  count = 0 # edit
  for question in list_questions:
    messages = [
        {"role": "system", "content": ("Hãy trả lời hoàn toàn bằng tiếng Việt. "
                                       "Dịch mọi từ/câu tiếng Anh hoặc ngoại ngữ khác trong phần trả lời sang tiếng Việt, "
                                       "trừ các nội dung cần để nguyên bản như tên riêng & mã nguồn. ")},
        {"role": "user", "content": question['content_turn1']}
    ]
    response1 = client.chat.completions.create(model = model, messages = messages)
    question['response_turn1'] = response1.choices[0].message.content

    messages.append({"role": "assistant", "content": response1.choices[0].message.content})
    messages.append({"role": "user", "content": question['content_turn2']})

    response2 = client.chat.completions.create(model = model, messages = messages)
    question['response_turn2'] = response2.choices[0].message.content

    print(f"Done question {count}")
    count += 1

    # Pause execution for 10 seconds
    time.sleep(10)
  return list_questions

In [ ]:
# Update index range for current run (e.g., continue from 80 to 99 after previous run covered 0–79)
model_question = copy.deepcopy(questions[80:])
# Adjust index to reference the correct model
model_answer = gen_model_answer(groq_models[0], groq_client, model_question) # edit
# model_answer = gen_model_answer(openrouter_models[1], openrouter_client, model_question) # edit

In [ ]:
transformed_data = []

for item in model_question:
    question_id = item.get('question_id')
    category = item.get('category')
    content_turn1 = item.get('content_turn1')
    response_turn1 = item.get('response_turn1')
    content_turn2 = item.get('content_turn2')
    response_turn2 = item.get('response_turn2')

    transformed_data.append({
        'question_id': question_id,
        'category': category,
        'content_turn1': content_turn1,
        'response_turn1': response_turn1,
        'content_turn2': content_turn2,
        'response_turn2': response_turn2
    })

df = pd.DataFrame(transformed_data)
df

In [ ]:
# Adjust index to reference the correct model
safe_model_name = groq_models[0].replace('/', '_') # edit
# safe_model_name = openrouter_models[1].replace('/', '_') # edit

# Adjust file path to match the target environment
output_file = f'drive/MyDrive/ViJudge/data/LLMs_responses/{safe_model_name}_response.csv'

if os.path.exists(output_file):
    # If file exists, append without writing header
    df.to_csv(output_file, mode='a', header=False, index=False)
else:
    # If file does not exist, write with header
    df.to_csv(output_file, mode='w', header=True, index=False)

print(f"Data has been written to file '{output_file}'")